In [1]:
# Install the required libraries
# This must run as the first cell
!pip uninstall -y huggingface_hub
!pip install -q -U \
    "transformers" \
    "datasets" \
    "accelerate" \
    "peft" \
    "trl" \
    "bitsandbytes" \
    "huggingface_hub" \
    "scikit-learn>=1.2,<1.9" \
    "google-cloud-bigquery-storage>=2.0.0"
!pip install -q -U "transformers @ git+https://github.com/huggingface/transformers.git@main"

Found existing installation: huggingface_hub 1.11.0
Uninstalling huggingface_hub-1.11.0:
  Successfully uninstalled huggingface_hub-1.11.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 101.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 47.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 107.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.0/306.0 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 79.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Verifying the environment
import sklearn
import transformers
import datasets
import accelerate
import peft
import trl
import bitsandbytes
import huggingface_hub

print("scikit-learn:", sklearn.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("Hugging Face Hub version:", huggingface_hub.__version__)
print("Loaded from:", huggingface_hub.__file__)

scikit-learn: 1.8.0
transformers: 5.15.0.dev0
datasets: 5.0.1
accelerate: 1.14.0
peft: 0.19.1
trl: 1.9.2
Hugging Face Hub version: 1.25.1
Loaded from: /usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py


# GPU Verification

Fine-tuning Qwen2.5-7B-Instruct in 4-bit requires a GPU.

This section verifies:

- Whether CUDA is available
- The GPU model assigned by Kaggle
- The available GPU memory
- The PyTorch version

**Note:** Qwen2.5-7B-Instruct is a 7-billion-parameter model. Even in 4-bit (QLoRA), it needs roughly 6-8GB of GPU memory for the base weights plus additional memory for gradients/activations/optimizer states. A single Kaggle T4 (16GB) can run it with a small per-device batch size (1) and gradient accumulation, and gradient checkpointing enabled. If available, enabling "GPU T4 x2" or a P100/A100 accelerator in the Kaggle notebook settings will give more headroom and faster training.


In [3]:
import os
import json
import torch
import pandas as pd

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


# Hugging Face Authentication

Unlike Gemma, `Qwen/Qwen2.5-3B-Instruct` is **not gated** on Hugging Face, so a token is not strictly required to download it.

However, it's still good practice to authenticate, since:

1. It avoids anonymous rate limits when pulling model weights.
2. The same token can be reused if you push the trained adapter back to the Hub.

Before running this section:

1. Create a Hugging Face access token.
2. Add the token to Kaggle Secrets using the name HF_TOKEN.
3. Enable the secret for this notebook.

The token is loaded securely from Kaggle Secrets and is not written directly inside the notebook.


In [4]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN was not found in Kaggle Secrets.")

login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token

# Locate the Dataset

Kaggle datasets are mounted under the following directory:

**/kaggle/input/**

This section lists all files attached to the notebook so that the exact CSV path can be identified.

The dataset used in this notebook is:

**curated_lms_tickets.csv** (from the `ranugaweerasekara/lms-dataset` Kaggle dataset)


In [5]:
import os
for dirname, _,filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ranugaweerasekara/new-dataset-lms/curated_lms_tickets.csv


# Load the Dataset

The customer-support ticket dataset is loaded using Pandas.

The dataset contains ticket information such as:

- Issue description
- Ticket category
- Priority
- Customer sentiment
- Product
- Communication channel
- Issue complexity score
- Resolution notes

The number of rows and columns is displayed to verify that the dataset was loaded correctly.

In [ ]:
# Let's load the dataset to using pandas
CSV_PATH = "C:\Users\ranug\Clario\clario\ml_finetuning\data\curated_synthetic_lms\curated_lms_tickets.csv"

df = pd.read_csv(CSV_PATH)

print("Shape: ", df.shape)
df.head()


Shape:  (20000, 13)


,ticket_id,category,issue_description,priority,sentiment,product,resolution_notes,status,channel,language,issue_complexity_score,platform,source
0,LMS-SYN-00001,Account Suspension,Account is suspended but I am just a student t...,Low,Neutral,Course Marketplace,Fixed database lock on user account.,Closed,Web Form,English,3,Rysera STEM LMS,gemini_synthetic_lms
1,LMS-SYN-00002,Account Suspension,Please restore my account. I have no idea why ...,Medium,Neutral,Student Mobile App,"Checked flag, restored mobile device access.",Resolved,Social Media,English,3,Rysera STEM LMS,gemini_synthetic_lms
2,LMS-SYN-00003,Subscription Cancellation,Can I cancel my subscription and keep access u...,Medium,Neutral,Learning Plan Subscription,Processed.,Resolved,Chat,English,2,Rysera STEM LMS,gemini_synthetic_lms
3,LMS-SYN-00004,Login Issue,Blocked from my account due to login issues.,High,Negative,Payment & Billing,Unblocked account following security review.,Resolved,Web Form,English,3,Rysera STEM LMS,gemini_synthetic_lms
4,LMS-SYN-00005,Performance Issue,"Everything is loading very slowly, is there a ...",Medium,Neutral,Learning Dashboard,Major outage resolved.,Resolved,Web Form,English,6,Rysera STEM LMS,gemini_synthetic_lms


# Explotary Dataset Analysis

Before fine-tuning the model, the dataset is inspected to understand its structure and quality.

This section examines:

- Column names
- Dataset dimensions
- Missing values
- Duplicate tickets
- Category distribution
- Priority distribution
- Sentiment distribution

This analysis helps identify possible class imbalance, missing information, and duplicate records.

---

## Dataset Features

The main columns used for fine-tuning are:

| Column                   | Purpose                                      |
| ------------------------ | -------------------------------------------- |
| `issue_description`      | Main customer-support message                |
| `category`               | Type of customer-support issue               |
| `priority`               | Urgency level of the ticket                  |
| `sentiment`              | Customer sentiment                           |
| `product`                | Product or service associated with the issue |
| `channel`                | Communication channel used by the customer   |
| `issue_complexity_score` | Estimated difficulty of resolving the issue  |
| `resolution_notes`       | Expected resolution or support action        |

Dataset metadata and identifier columns are excluded because they do not provide meaningful information for ticket prediction.

In [7]:
df.columns.tolist()

['ticket_id',
 'category',
 'issue_description',
 'priority',
 'sentiment',
 'product',
 'resolution_notes',
 'status',
 'channel',
 'language',
 'issue_complexity_score',
 'platform',
 'source']

In [8]:
print("Misssing values:")
print(df.isnull().sum())
# There is no missing values in the dataset

Misssing values:
ticket_id                 0
category                  0
issue_description         0
priority                  0
sentiment                 0
product                   0
resolution_notes          0
status                    0
channel                   0
language                  0
issue_complexity_score    0
platform                  0
source                    0
dtype: int64


In [9]:
print("Duplicate values:")
print(df.duplicated().sum())
# there is no duplicate rows in the dataset

Duplicate values:
0


In [10]:
print("Categories:")
df["category"].value_counts()
# For category column data is balanced

Categories:


category
Account Suspension           2500
Subscription Cancellation    2500
Login Issue                  2500
Performance Issue            2500
Feature Request              2500
Bug Report                   2500
Refund Request               2500
Payment Problem              2500
Name: count, dtype: int64

In [11]:
print("Priorities:")
df["priority"].value_counts()

Priorities:


priority
Medium    8271
High      5909
Low       4365
Urgent    1455
Name: count, dtype: int64

In [12]:
print("Sentiments:")
df["sentiment"].value_counts()

Sentiments:


sentiment
Negative             8981
Neutral              7672
Strongly Negative    2372
Positive              975
Name: count, dtype: int64

# Dataset Cleaning

The dataset is cleaned before training to improve model quality.

The cleaning process includes:

* Removing duplicate ticket descriptions
* Selecting only useful columns
* Converting text columns to strings
* Removing unnecessary whitespace
* Inspecting very short resolution notes

Duplicate descriptions must be removed before splitting the dataset. Otherwise, similar examples may appear in both the training and test sets, resulting in misleadingly high evaluation scores.

---

## Resolution Note Quality

Some resolution notes may be too short or vague, for example:

```text
Processed.
```

These responses may not teach the model to generate helpful customer-support resolutions.

For the initial experiment, all records may be retained. In later experiments, low-quality resolution notes can be removed or manually improved.



In [13]:
print(
    "Duplicate descriptions:",
    df.duplicated(subset=["issue_description"]).sum()
)

Duplicate descriptions: 7


In [14]:
# Since there 7 duplicated issue_description we have to remove them

df = df.drop_duplicates(
    subset=["issue_description"]
).reset_index(drop=True)

In [15]:
selected_columns = [
    "issue_description",
    "category",
    "priority",
    "sentiment",
    "product",
    "resolution_notes",
    "issue_complexity_score"
]

df = df[selected_columns].copy()

In [16]:
df.head()

,issue_description,category,priority,sentiment,product,resolution_notes,issue_complexity_score
0,Account is suspended but I am just a student t...,Account Suspension,Low,Neutral,Course Marketplace,Fixed database lock on user account.,3
1,Please restore my account. I have no idea why ...,Account Suspension,Medium,Neutral,Student Mobile App,"Checked flag, restored mobile device access.",3
2,Can I cancel my subscription and keep access u...,Subscription Cancellation,Medium,Neutral,Learning Plan Subscription,Processed.,2
3,Blocked from my account due to login issues.,Login Issue,High,Negative,Payment & Billing,Unblocked account following security review.,3
4,"Everything is loading very slowly, is there a ...",Performance Issue,Medium,Neutral,Learning Dashboard,Major outage resolved.,6


In [17]:
# Clean text values: Remove white spaces if theres any
text_columns = [
    "issue_description",
    "category",
    "priority",
    "sentiment",
    "product",
    "resolution_notes"
]

for column in text_columns:
    df[column] = df[column].astype(str).str.strip()

In [18]:
# Check short resolution notes
short_resolutions = df[
    df["resolution_notes"].str.len() < 15
]
print("Very short resolutions:", len(short_resolutions))
short_resolutions.head()

# Optional for response-generation experiments
# df = df[df["resolution_notes"].str.len() >= 15].reset_index(drop=True)

Very short resolutions: 733


,issue_description,category,priority,sentiment,product,resolution_notes,issue_complexity_score
2,Can I cancel my subscription and keep access u...,Subscription Cancellation,Medium,Neutral,Learning Plan Subscription,Processed.,2
27,Cancel my subscription plan. I need to maintai...,Subscription Cancellation,Medium,Neutral,Learning Plan Subscription,Processed.,1
31,I need help with my subscription cancellation....,Subscription Cancellation,Medium,Neutral,Learning Plan Subscription,Processed.,3
41,Paid for the annual subscription but realized ...,Refund Request,Low,Neutral,Learning Plan Subscription,Refund issued.,2
52,Cancel my sub but keep access until the end of...,Subscription Cancellation,Low,Neutral,Learning Plan Subscription,Processed.,1


# Train, Validation and Test Split

The dataset is divided into three subsets:

* **80% training data**
* **10% validation data**
* **10% testing data**

The training set is used to update the LoRA adapter.

The validation set is used to monitor model performance during training and select the best checkpoint.

The test set is kept unseen during training and is used for final evaluation.

The dataset is stratified using the `category` column. This ensures that all ticket categories are represented proportionally in the training, validation and testing sets.


In [19]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size = 0.20,
    random_state=42,
    stratify = df["category"]
)

validation_df,test_df = train_test_split(
    temp_df,
    test_size = 0.50,
    random_state = 42,
    stratify = temp_df["category"]
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Traning rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Testing rows:", len(test_df))

Traning rows: 15994
Validation rows: 1999
Testing rows: 2000


In [20]:
# Verify the destribution
def compare_distributions(column):
    comparison = pd.DataFrame({
        "Full Dataset": df[column].value_counts(normalize=True),
        "Training": train_df[column].value_counts(normalize=True),
        "Validation": validation_df[column].value_counts(normalize=True),
        "Testing": test_df[column].value_counts(normalize=True)
    }).fillna(0) * 100

    return comparison.round(2)

display(compare_distributions("category"))
display(compare_distributions("priority"))
display(compare_distributions("sentiment"))

,Full Dataset,Training,Validation,Testing
category,,,,
Account Suspension,12.50,12.50,12.51,12.5
Bug Report,12.49,12.49,12.46,12.5
Feature Request,12.50,12.50,12.51,12.5
Login Issue,12.50,12.50,12.51,12.5
Payment Problem,12.50,12.50,12.51,12.5
Performance Issue,12.49,12.49,12.51,12.5
Refund Request,12.50,12.50,12.51,12.5
Subscription Cancellation,12.50,12.50,12.51,12.5


,Full Dataset,Training,Validation,Testing
priority,,,,
Medium,41.35,41.32,41.47,41.55
High,29.55,29.55,29.31,29.70
Low,21.82,22.01,20.81,21.30
Urgent,7.28,7.12,8.40,7.45


,Full Dataset,Training,Validation,Testing
sentiment,,,,
Negative,44.90,44.79,45.77,44.85
Neutral,38.36,38.60,37.02,37.80
Strongly Negative,11.86,11.81,12.46,11.70
Positive,4.88,4.80,4.75,5.65


# Define the System Prompt

The system prompt describes the role of the model and specifies the expected response format.

The model is instructed to:

* Analyze a customer-support ticket
* Select values from predefined labels
* Produce one valid JSON object
* Generate a concise resolution
* Avoid additional explanations or Markdown formatting

A consistent system prompt improves output stability and makes model responses easier to parse.


In [21]:
SYSTEM_PROMPT = """
You are an AI assistant for a Learning Management System support team.

Analyze the supplied customer-support ticket and return one valid JSON object.

The JSON must contain exactly these keys:
category, priority, sentiment, resolution.

Allowed categories:
Account Suspension,
Subscription Cancellation,
Login Issue,
Performance Issue,
Feature Request,
Bug Report,
Refund Request,
Payment Problem.

Allowed priorities:
Low, Medium, High, Urgent.

Allowed sentiments:
Positive, Neutral, Negative, Strongly Negative.

The resolution must be short, helpful and directly related to the ticket.
Do not output markdown or explanations outside the JSON object.
""".strip()

# Convert Records into Conversations

Qwen2.5-Instruct is an instruction-tuned conversational model.

Each dataset row is converted into a conversation containing three messages:

1. **System message** — explains the model's task
2. **User message** — contains the support-ticket details
3. **Assistant message** — contains the expected JSON response

Example training conversation:

```text
System:
You are an AI assistant for a Learning Management System support team.

User:
Ticket description: I cannot log in to my student account.
Product: Student Portal
Channel: Email
Issue complexity score: 4

Assistant:
{
  "category": "Login Issue",
  "priority": "Medium",
  "sentiment": "Negative",
  "resolution": "Reset the user's login credentials and ask them to sign in again."
}
```

This format teaches Qwen2.5 how to respond to similar instructions during inference. `SFTTrainer` applies the tokenizer's built-in chat template automatically to the `messages` column, so the exact formatting matches how Qwen2.5-Instruct was trained (ChatML-style special tokens).


In [22]:
def create_messages(row):
    user_content = (
        f"Ticket description: {row['issue_description']}\n"
        f"Product: {row['product']}\n"
        f"Issue complexity score: {row['issue_complexity_score']}"
    )

    assistent_content = json.dumps(
        {
            "category": row["category"],
            "priority": row["priority"],
            "sentiment": row["sentiment"],
            "resolution": row["resolution_notes"]
        },
        ensure_ascii = False
    )

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_content
        },
        {
            "role": "assistant",
            "content": assistent_content
        }
    ]

# Create Hugging Face Datasets

The Pandas DataFrames are converted into Hugging Face `Dataset` objects.

The following dataset splits are created:

* `train`
* `validation`
* `test`

A `messages` column is added to every record. This column contains the conversational representation required by the supervised fine-tuning trainer.

Unnecessary original columns may be removed after conversation creation to reduce memory usage.



In [23]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df,
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        validation_df,
        preserve_index=False
    ),
    "test": Dataset.from_pandas(
        test_df,
        preserve_index=False
    )
})

In [24]:
def add_messages(example):
    return {
        "messages": create_messages(example)
    }

dataset = dataset.map(add_messages)
dataset

Map:   0%|          | 0/15994 [00:00<?, ? examples/s]

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['issue_description', 'category', 'priority', 'sentiment', 'product', 'resolution_notes', 'issue_complexity_score', 'messages'],
        num_rows: 15994
    })
    validation: Dataset({
        features: ['issue_description', 'category', 'priority', 'sentiment', 'product', 'resolution_notes', 'issue_complexity_score', 'messages'],
        num_rows: 1999
    })
    test: Dataset({
        features: ['issue_description', 'category', 'priority', 'sentiment', 'product', 'resolution_notes', 'issue_complexity_score', 'messages'],
        num_rows: 2000
    })
})

In [25]:
dataset["train"][0]["messages"]

[{'role': 'system',
  'content': 'You are an AI assistant for a Learning Management System support team.\n\nAnalyze the supplied customer-support ticket and return one valid JSON object.\n\nThe JSON must contain exactly these keys:\ncategory, priority, sentiment, resolution.\n\nAllowed categories:\nAccount Suspension,\nSubscription Cancellation,\nLogin Issue,\nPerformance Issue,\nFeature Request,\nBug Report,\nRefund Request,\nPayment Problem.\n\nAllowed priorities:\nLow, Medium, High, Urgent.\n\nAllowed sentiments:\nPositive, Neutral, Negative, Strongly Negative.\n\nThe resolution must be short, helpful and directly related to the ticket.\nDo not output markdown or explanations outside the JSON object.'},
 {'role': 'user',
  'content': 'Ticket description: System says I am already logged in on another device.\nProduct: Video Classroom\nIssue complexity score: 3'},
 {'role': 'assistant',
  'content': '{"category": "Login Issue", "priority": "Low", "sentiment": "Neutral", "resolution": 

In [26]:
# remove the original columns to reduce memory
columns_to_remove = [
    column
    for column in dataset["train"].column_names
    if column != "messages"
]

dataset = dataset.remove_columns(columns_to_remove)
dataset

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 15994
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 1999
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 2000
    })
})

# Load Llama-3.1-8B-Instruct

The base model used in this notebook is:

```text
Llama-3.1-8B-Instruct
```

The `8B-Instruct` checkpoint is selected because:

* It is instruction-tuned and handles structured JSON output well
* It has strong general reasoning ability for classifying tickets
* It fits on a single Kaggle T4 GPU when loaded in 4-bit with QLoRA
* It is not gated, so no license acceptance is required

The model is loaded using 4-bit quantization to reduce GPU memory usage.

---

## 4-Bit Quantization

The model is loaded using the Normal Float 4-bit format.

The quantization configuration uses:

* 4-bit model weights
* NF4 quantization
* Double quantization
* Float16 or BFloat16 computation

Quantization reduces the memory required to store the base model while preserving most of its original capability.


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"


## Configure QLoRA:

In [44]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
else:
    compute_dtype = torch.float16

print("Compute dtype:", compute_dtype)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

Compute dtype: torch.bfloat16


## Load the tokenizer:

In [45]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=hf_token
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [46]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    quantization_config=quantization_config,
    dtype=compute_dtype,
    device_map={"": 0}
)

model.config.use_cache = False
model.config.pretraining_tp = 1


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

# Prepare the Model for QLoRA Training

The quantized model must be prepared before LoRA adapters are attached.

This preparation process:

* Freezes the original model parameters
* Enables gradient checkpointing
* Prepares selected layers for low-bit training
* Reduces GPU memory consumption

Only the LoRA adapter parameters will be updated during training.

In [47]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

# 14. Configure LoRA

Low-Rank Adaptation introduces small trainable matrices into selected transformer layers.

The LoRA adapter is applied to:

* Query projection
* Key projection
* Value projection
* Output projection
* Gate projection
* Up projection
* Down projection

These module names are the same for Qwen2.5's transformer blocks as they were for Gemma 3, so the LoRA configuration below still applies directly.

The main LoRA settings are:

| Parameter | Value | Description                    |
| --------- | ----: | ------------------------------ |
| Rank      |    16 | Capacity of the LoRA adapter   |
| Alpha     |    32 | Scaling factor                 |
| Dropout   |  0.05 | Regularization                 |
| Bias      |  None | Does not train bias parameters |

LoRA allows the model to learn the ticket-analysis task without updating every parameter in Qwen2.5-7B.


In [48]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM",
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

# Training Configuration

Qwen2.5-7B is much larger than Gemma-3-1B, so the batch size and gradient accumulation are adjusted to fit a single T4 GPU (16GB) even in 4-bit:

| Parameter               |             Value |
| ----------------------- | ----------------: |
| Epochs                  |                 3 |
| Training batch size     |                 1 |
| Validation batch size   |                 1 |
| Gradient accumulation   |                16 |
| Effective batch size    |                16 |
| Learning rate           |            0.0002 |
| Maximum sequence length |               512 |
| Optimizer               | Paged AdamW 8-bit |
| Learning-rate scheduler |            Cosine |
| Weight decay            |              0.01 |

Gradient accumulation is used to simulate a larger batch size without exceeding GPU memory.

The effective batch size is calculated as:

```text
Training batch size × Gradient accumulation steps
1 × 16 = 16
```

If you have access to a larger GPU (e.g. Kaggle "GPU T4 x2", P100, or an A100), you can raise `per_device_train_batch_size` to 2 and lower `gradient_accumulation_steps` accordingly to speed up training while keeping the same effective batch size.


In [49]:
from trl import SFTConfig

OUTPUT_DIR = "/kaggle/working/Qwen-3B-Instruct-lms-ticket_adapter"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=0.05,
    lr_scheduler_type="cosine",

    logging_steps=20,

    eval_strategy="steps",
    eval_steps=100,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    max_length=512,

    gradient_checkpointing=True,

    fp16=compute_dtype==torch.float16,
    bf16=compute_dtype==torch.bfloat16,

    optim="paged_adamw_8bit",

    report_to="none",
    seed=42
)


## Create the Trainer

In [50]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    processing_class=tokenizer
)

Tokenizing train dataset:   0%|          | 0/15994 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/15994 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15994 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1999 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1999 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1999 [00:00<?, ? examples/s]

In [51]:
trainer.model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


# Small Pipeline Test

Before running the full training process, a small test run is performed using a subset of the dataset.

The purpose of this test is to verify that:

* The model loads correctly
* The dataset format is valid
* The trainer accepts the configuration
* Gradients are calculated correctly
* GPU memory is sufficient
* Model checkpoints can be saved

A short test run prevents wasting Kaggle GPU time on a configuration error.

After the small test succeeds, the Kaggle session should be restarted before beginning the complete training run.

In [52]:
small_train_dataset = dataset["train"].select(
    range(min(500, len(dataset["train"])))
)

small_validation_dataset = dataset["validation"].select(
    range(min(100, len(dataset["validation"])))
)

In [53]:
test_training_args = SFTConfig(
    output_dir="/kaggle/working/qwen2.5-3b-test-run",

    max_steps=20,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=5,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="no",

    max_length=512,
    gradient_checkpointing=True,

    fp16=compute_dtype == torch.float16,
    bf16=compute_dtype == torch.bfloat16,

    optim="paged_adamw_8bit",
    report_to="none"
)


In [54]:
test_trainer = SFTTrainer(
    model=model,
    args=test_training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_validation_dataset,
    peft_config=peft_config,
    processing_class=tokenizer
)

test_trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10,2.624269,2.353005,1.046879,5914.000000,0.770125
20,2.261319,2.181292,0.954064,11839.000000,0.774302


TrainOutput(global_step=20, training_loss=3.0380208015441896, metrics={'train_runtime': 506.1081, 'train_samples_per_second': 0.316, 'train_steps_per_second': 0.04, 'total_flos': 530412728795136.0, 'train_loss': 3.0380208015441896, 'epoch': 0.32})

# Full Model Training

The full supervised fine-tuning process is started in this section.

During training, the notebook records:

* Training loss
* Validation loss
* Training steps
* Learning rate
* Runtime
* Model checkpoints

The best checkpoint is selected using validation loss.

A decreasing training and validation loss generally indicates that the model is learning the required task.

If the training loss decreases while validation loss increases, the model may be overfitting.


In [ ]:
train_result = trainer.train()

FINAL_ADAPTER_PATH = (
    "/kaggle/working/qwen-3b-lms-ticket-adapter-final"
)

trainer.save_model(FINAL_ADAPTER_PATH)
tokenizer.save_pretrained(FINAL_ADAPTER_PATH)


Step,Training Loss,Validation Loss


In [ ]:
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

In [ ]:
# =========================================================
# Evaluate the fine-tuned model on the held-out 10% test set
# Run this in the SAME session, right after training —
# model, tokenizer, and the splits are already in memory.
# =========================================================
import re
import json
import torch
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

model.eval()
model.config.use_cache = True  # speeds up generation

def build_prompt(row):
    user_content = (
        f"Ticket description: {row['issue_description']}\n"
        f"Product: {row['product']}\n"
        f"Issue complexity score: {row['issue_complexity_score']}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content}
    ]

def generate_prediction(row):
    messages = build_prompt(row)
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=200,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = output_ids[0][input_ids.shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return text

def parse_prediction(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return {"category": "INVALID", "priority": "INVALID", "sentiment": "INVALID", "resolution": ""}
    try:
        obj = json.loads(match.group(0))
        return {
            "category": obj.get("category", "INVALID"),
            "priority": obj.get("priority", "INVALID"),
            "sentiment": obj.get("sentiment", "INVALID"),
            "resolution": obj.get("resolution", "")
        }
    except json.JSONDecodeError:
        return {"category": "INVALID", "priority": "INVALID", "sentiment": "INVALID", "resolution": ""}

predictions = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating predictions"):
    raw_text = generate_prediction(row)
    predictions.append(parse_prediction(raw_text))

pred_df = pd.DataFrame(predictions)

def print_report(field_name, y_true, y_pred):
    labels = sorted(set(y_true) | set(y_pred))
    print(field_name.upper())
    print("Accuracy:", accuracy_score(y_true, y_pred))
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro", zero_division=0
    )
    print("Macro F1:", f1)
    print("Macro Precision:", precision)
    print("Recall:", recall)
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    print()

print_report("category", test_df["category"].tolist(), pred_df["category"].tolist())
print_report("priority", test_df["priority"].tolist(), pred_df["priority"].tolist())
print_report("sentiment", test_df["sentiment"].tolist(), pred_df["sentiment"].tolist())